# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [3]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [4]:
# Task 1 — Treemap: fossil fuel TWh by Region -> Country -> Source

# ── Data prep ─────────────────────────────────────────────────────────────────
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

region_tot = df_fossil.groupby('Region')['TWh'].sum().sort_values(ascending=False)
top_region = region_tot.index[0]
top_region_coal = df_fossil.loc[(df_fossil['Region'] == top_region) & (df_fossil['Source'] == 'Coal'), 'TWh'].sum()

# CVD-safe colours (Okabe-Ito palette) — one per fossil source
fossil_colours = {
    'Coal':        '#D55E00',   # vermillion
    'Oil':         '#0072B2',   # blue
    'Natural Gas': '#009E73',   # bluish green
}

# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colours,
    height=700, width=1100,
    template='plotly_white'
)

# ── Step 2: Customisation ─────────────────────────────────────────────────────
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',          # TWh only, no percentages
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
)

# Grey out parent nodes (Region and Country) — keep colour only on the sources
fig.data[0].marker.colors = [
    c if c in fossil_colours.values() else '#DDDDDD'
    for c in fig.data[0].marker.colors
]

fig.update_layout(
    title=(f'{top_region} burns the most fossil energy ({region_tot.iloc[0]:,.0f} TWh) — '
           f'<span style="color:#D55E00"><b>{top_region_coal / region_tot.iloc[0]:.0%} of it is coal</b></span>'),
    font=dict(family='Arial', size=12),
)
fig.show()


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [7]:
# Task 2 — Low-carbon (Nuclear + Hydro) TWh per country: treemap vs bar

# ── Data prep ─────────────────────────────────────────────────────────────────
low = (df.loc[df['Source_Type'] == 'Low-carbon']
         .groupby('Country')['TWh'].sum()
         .reset_index()
         .sort_values('TWh', ascending=False))
low['All'] = 'Low-carbon'                      

# ── Treemap ───────────────────────────────────────────────────────────────────
fig_tree = px.treemap(
    low,
    path=['All', 'Country'],
    values='TWh',
    height=550, width=1000,
    template='plotly_white'
)
fig_tree.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    marker_line_color='white', marker_line_width=2,
)
# Grey root node, blue countries (CVD-safe blue, as in the notes)
fig_tree.data[0].marker.colors = [
    '#DDDDDD' if node_id == 'Low-carbon' else '#2E75B6'
    for node_id in fig_tree.data[0].ids
]
fig_tree.update_layout(title='Low-carbon energy (Nuclear + Hydro) by country', font=dict(family='Arial', size=12))
fig_tree.show()

# ── Bar chart ─────────────────────────────────────────────────────────────────
leader = low.iloc[0]
runner = low.iloc[1]

fig_bar = px.bar(
    low,
    x='TWh', y='Country',
    orientation='h',
    text='TWh',
    color_discrete_sequence=['#2E75B6'],       # CVD-safe blue (same as the notes)
    category_orders={'Country': low['Country'].tolist()},   # largest at the top
    labels={'TWh': 'Low-carbon energy (TWh)', 'Country': ''},
    height=550, width=1000,
    template='plotly_white'
)
fig_bar.update_traces(texttemplate='%{text:,.0f} TWh', textposition='outside')
fig_bar.update_layout(
    title=(f"{leader['Country']} leads low-carbon energy with {leader['TWh']:,.0f} TWh — "
           f"{leader['TWh'] - runner['TWh']:,.0f} TWh ahead of {runner['Country']}"),
    font=dict(family='Arial', size=12),
    xaxis=dict(range=[0, low['TWh'].max() * 1.15], showgrid=False),
)
fig_bar.show()


## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [8]:
# Task 3 — 100% stacked bar: coal share of each region's fossil energy

# ── Data prep ─────────────────────────────────────────────────────────────────
reg = df_fossil.groupby(['Region', 'Source'])['TWh'].sum().reset_index()
reg['Share'] = reg['TWh'] / reg.groupby('Region')['TWh'].transform('sum')   # share of region's fossil total

# Coal share per region — sorted, highest at the top
coal = (reg.loc[reg['Source'] == 'Coal', ['Region', 'Share']]
           .set_index('Region')['Share']
           .reindex(reg['Region'].unique(), fill_value=0)       # regions with no coal get 0
           .sort_values(ascending=False))
region_order = coal.index.tolist()

# Labels on coal segments only
reg['Label'] = np.where(reg['Source'] == 'Coal', reg['Share'].map('{:.0%}'.format), '')

# Coal = same colour as Task 1; Oil and Gas in two greys
accent_mapping = {
    'Coal':        '#D55E00',   # accent — the story
    'Oil':         '#BBBBBB',   # mid grey — context
    'Natural Gas': '#DDDDDD',   # light grey — context
}

# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig = px.bar(
    data_frame=reg,
    x='Share', y='Region',
    color='Source',
    text='Label',
    color_discrete_map=accent_mapping,
    category_orders={'Region': region_order,
                     'Source': ['Coal', 'Oil', 'Natural Gas']},   # coal on the baseline
    labels={'Share': 'Share of fossil energy', 'Source': 'Fossil source'},
    hover_name='Region',
    hover_data={'Label': False, 'Share': ':.0%', 'TWh': ':,.0f'},
    height=550, width=1000,
    template='plotly_white',
)

# ── Step 2: Customisation ─────────────────────────────────────────────────────
fig.update_traces(
    textposition='inside', insidetextanchor='start',
    textfont=dict(color='white', size=12),
    marker_line_color='white', marker_line_width=1,
)
fig.update_layout(
    title=f'{region_order[0]} relies most on coal — {coal.iloc[0]:.0%} of its fossil energy comes from coal',
    font=dict(family='Arial', size=12),
    bargap=0.25,
    legend=dict(orientation='h', x=0, y=1.08, title=''),
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(title=''),
)
fig.add_annotation(
    x=1, y=-0.12, xref='paper', yref='paper', showarrow=False, xanchor='right',
    text='Each bar = 100% of the region\'s fossil energy (TWh)',
    font=dict(size=11, color='grey'),
)
fig.show()
